In [1]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Iterable

In [2]:
WORD_RE = re.compile(r"\b\w+\b", flags=re.UNICODE)

def iter_files(folder: Path, recursive: bool) -> Iterable[Path]:
    if recursive:
        yield from (p for p in folder.rglob("*") if p.is_file())
    else:
        yield from (p for p in folder.iterdir() if p.is_file())

def count_words_in_text(text: str) -> int:
    return len(WORD_RE.findall(text))

def read_text_safely(path: Path) -> str | None:
    # Skip hidden files (optional, but often helpful)
    if path.name.startswith("."):
        return None

    # Try common encodings
    for enc in ("utf-8", "latin-1"):
        try:
            return path.read_text(encoding=enc, errors="strict")
        except UnicodeDecodeError:
            continue
        except OSError:
            return None
    # If still failing, last resort: decode with replacement
    try:
        return path.read_text(encoding="utf-8", errors="replace")
    except OSError:
        return None

def count_words_in_folder(folder_path: str, recursive: bool = True) -> int:
    folder = Path(folder_path)
    if not folder.exists() or not folder.is_dir():
        raise ValueError(f"Folder not found or not a directory: {folder}")

    total = 0
    for file_path in iter_files(folder, recursive=recursive):
        text = read_text_safely(file_path)
        if text is None:
            continue
        total += count_words_in_text(text)

    return total

In [4]:
news_folder = "../data/final-news"
total_words = count_words_in_folder(news_folder, recursive=True)

print(f"Total words in news articles: {total_words}")

Total words in news articles: 9205199
